# 07. Surrogate Model Training (MLP)

This notebook trains a **Neuro-FuzzyGÇôinspired neural surrogate model** to approximate the ANFIS reasoning layer.
While a classical ANFIS consists of explicit membership functions and fuzzy rules, this surrogate neural model learns a smooth mapping that emulates the same reasoning behaviour for real-time deployment in the game engine.

## 1. Setup and Imports

In [ ]:
# Install necessary libraries if not present
!pip install scikit-learn pandas numpy

In [ ]:
import pandas as pd
import os
import json
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Inputs & Outputs
INPUT_PATH = '../../data/processed/6_anfis_dataset.csv'
MODEL_OUT_PATH = '../../data/models/anfis_params.json'

print("Libraries imported successfully.")

## 2. Load Data

In [ ]:
print("Loading dataset...")
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(df)} rows.")
else:
    raise FileNotFoundError(f"File not found at {INPUT_PATH}")

## 3. Prepare Training and Test Sets

In [ ]:
# Features & Target
X = df[['pct_combat', 'pct_collect', 'pct_explore', 'delta_combat', 'delta_collect', 'delta_explore']]
y = df['target_multiplier']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Set: {X_train.shape}")
print(f"Test Set: {X_test.shape}")

## 4. Train MLP Regressor
We use a compact architecture (16, 8) to ensure runtime efficiency in the game engine.

In [ ]:
print("Initializing and training MLP Regressor...")
# 6 Inputs -> 16 Hidden -> 8 Hidden -> 1 Output
mlp = MLPRegressor(hidden_layer_sizes=(16, 8), activation='relu', solver='adam', max_iter=1000, random_state=42)
mlp.fit(X_train, y_train)
print("Training complete.")

## 5. Model Evaluation

In [ ]:
# Evaluate
y_predict = mlp.predict(X_test)
mse = mean_squared_error(y_test, y_predict)
mae = mean_absolute_error(y_test, y_predict)

print("--- Evaluation Metrics ---")
print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")

## 6. Export Model Parameters
We extract weights and biases to a JSON file for import into the runtime game engine inference layer.

In [ ]:
# Accessing weights: coefs_ (weights), intercepts_ (biases)
params = {
    'layer_1_weights': mlp.coefs_[0].tolist(),
    'layer_1_bias': mlp.intercepts_[0].tolist(),
    'layer_2_weights': mlp.coefs_[1].tolist(),
    'layer_2_bias': mlp.intercepts_[1].tolist(),
    'output_layer_weights': mlp.coefs_[2].tolist(),
    'output_layer_bias': mlp.intercepts_[2].tolist(),
    'activation': mlp.activation
}

os.makedirs('../../data/models', exist_ok=True)
with open(MODEL_OUT_PATH, 'w') as f:
    json.dump(params, f, indent=4)
    
print(f"Model parameters exported to: {MODEL_OUT_PATH}")